# 00 · SHAP attribution across FTTL versions — real data

**What this notebook does NOT do: it never opens a model pickle.** A pickle only unpickles inside
the env it was serialised in, and the three FTTL versions have mutually incompatible library stacks
(`src/docs/ENV_MANAGEMENT.md`). So the split is:

| step | where it runs | what it produces |
|---|---|---|
| compute per-row φ | `src/envs/v<k>/.venv/bin/python src/attribution/attribute.py` — **once per version, in that version's own env** | `src/data/real/detection/shap/<v>_attributions_<split>.parquet` (+ `_<suffix>` for a second backend on the same split) + `..._meta.json` |
| **this notebook** | the shared analysis `.venv` (kernel `ml-sfp-detection`) | tables + figures, for THREE configurations at once |

Run the compute step first — one command block per configuration this notebook reads (§0 below
lists exactly which), once per version since there is no all-versions driver:

```bash
src/envs/v2/.venv/bin/python src/attribution/attribute.py --model ... --features ... \
    --version v2 --split train --rows 5000 --background 500 --out ..._attributions_train.parquet
```

Repeat with `--version`/`--features`/`--out` swapped for v1 and v3; for each version's own
holdout (v1 `val2` · v2 `test` · v3 `oot`); and once more on that holdout with `--backend native`
and a `_native`-suffixed `--out` path.

**Which split, and why it is named.** The φ files exist only per split. Split names are each
version's own — v2's holdout is `test`, v3's is `oot`, v1 has `val1`/`val2` — and they are never
unified. It is load-bearing: concentration measured on `train` describes the fitted function on
data it saw, on a holdout it describes generalisation, and comparing one against the other is a
confound, not a finding — which is exactly the comparison §0 below sets up on purpose, tagged as
such rather than left implicit.

φ values are just numbers once they are on disk, so everything below is version-agnostic and the
dependency problem disappears. Nothing here hard-codes a path, a repo name or a column name —
they all come from `src/config.py` via `loaders.load()`.

**Read every number below at the ENCODED feature names** — each model's own
post-preprocessing columns, exactly as the booster sees them, never collapsed back to the
concept they encode. v1's 55
`make_*` columns and v2's 41 are *not* collapsed into one `make` — doing so would change the
concentration statistic this chapter rests on. Cross-version correspondence comes **only** from the
hand-confirmed mapping (`features/check_overlap.py` → `features/feature_overlap.json`), never from
name equality.

**Figure naming convention (matches `00_SHAP.ipynb`).** Every figure is named
`<section><letter>_<description>`, where `<section>` is the two-digit markdown section number it
sits under and `<letter>` starts at `a` within that section — even when the section has only one
figure — and increments for each further figure the section produces. The figures in this notebook are
`01a_real_shap_global_importance_{run_label}` (§1), `04<letter>_real_shap_hill_profile_by_run_<basis>` (§4 —
one per comparison basis; letters fixed by basis order so a skipped basis leaves a gap rather than
shifting the rest) and `07a_real_shap_backend_shift` (§7). A figure added later reuses the next free letter in its
own section rather than a separate numbering scheme. **01a and 07a are the only two figures that
label individual features, so each also saves an `alias_`-prefixed twin** (real Allianz column
names swapped for `feature_alias.to_alias()` aliases) — see the Notes section at the bottom.

In [ ]:
import sys
import json
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
# Without this the loop ends at "/" on a mis-started kernel, puts "/src" on the path, and the
# failure surfaces as ModuleNotFoundError two cells later, naming config rather than the cwd.
assert (ROOT / "src" / "config.py").exists(), (
    f"repo root not found above {Path.cwd()} — start the kernel inside the repo.")
sys.path.insert(0, str(ROOT / "src"))

import config           # noqa: E402  — real paths/columns live here and nowhere else
import schema           # noqa: E402
import figstyle         # noqa: E402
import feature_alias    # noqa: E402  — real feature names never go on an axis/legend, only aliases
from loaders import load                       # noqa: E402
from estimator import concentration as conc    # noqa: E402

figstyle.apply()

SOURCE   = "real"
VERSIONS = list(config.VERSION_LABELS)   # trim to e.g. ["v2", "v3"] to compare a single pair

TOPN = 15    # features shown per version in the importance panels
TOPK = 5     # k for top-k share

print(f"repo root : {ROOT}")
print(f"versions  : {', '.join(VERSIONS)}  (source={SOURCE})")


## 0 · The three configurations this notebook compares

Every section below runs three times — once per row — instead of once for a single hard-coded
split:

| run | split | backend / perturbation | question it answers |
|---|---|---|---|
| `interventional_train` | each version's `"train"` split — the one split name spelled identically across all three (`config.SPLITS`) | `shap`, interventional, fixed background | concentration on data the fit saw |
| `interventional_oot` | each version's own out-of-time holdout (`config.OOT_SPLIT`: v1 `val2`, v2 `test`, v3 `oot`) | `shap`, interventional | concentration on data the fit did NOT see — this notebook's previous (and only) default |
| `path_dependent` | the SAME split as `interventional_oot` | `native` — each estimator's own TreeSHAP, tree-path-dependent | isolates the BACKEND as the only variable: does the reference/perturbation choice itself move the concentration numbers, split held fixed? |

`path_dependent` deliberately reuses `interventional_oot`'s split rather than a split of its own —
comparing it to `interventional_oot` is only a clean backend-only comparison if the split does not
also change. (A path-dependent run on `train` is one line away — call `analyze()` again with
`TRAIN_SPLITS` and `out_suffix="_native_train"`.)

**Why `path_dependent` needs `--out-suffix`.** `config.path("attributions", ...)` is keyed only by
`(version, split)`, not by backend — so attributing OOT under `native` after already attributing it
under `shap` would silently overwrite the first parquet at the identical path, so the caller
appends a suffix to `--out` by hand for exactly this case. The `path_dependent` run below reads that suffixed file
by an explicit path, bypassing `loaders.load()`'s normal (unsuffixed) resolution — for
`.attributions`/`.attribution_meta` only. `.frame`/`.decisions`/`.tau` (scores, targets, the decision
rule) do not depend on the SHAP backend and still resolve the normal way through the same
`VersionData` object.


In [ ]:
def splits_for(spec) -> dict[str, str]:
    """Resolve a run's split declaration into {version: split}.

    `spec` is either ONE name every version spells identically ("train") or an explicit
    {version: split} map. Split names are each version's OWN and deliberately not unified
    (v1 train/test/val1/val2 · v2 train/val/test · v3 train/test/oot), so "the validation
    split" does not exist across all three — spell those per version.

    A version the spec omits, or one whose SPLITS lack that name, is DROPPED from the run
    with a printed reason rather than raising: v3 has no validation split at all, and a run
    over the two versions that do have one is still worth having.

    This is the notebook twin of `config.resolve_splits()`, which takes the SAME two forms as
    the CLI `--split` argument but EXITS on a name a version does not have. The policies differ
    on purpose: the CLI must catch a typo before it launches per-version subprocesses, whereas
    here a two-version run is a legitimate result. Every drop is therefore printed — a run that
    silently shrinks from three versions to two would otherwise be read as a three-version one.
    """
    wanted = {v: spec for v in VERSIONS} if isinstance(spec, str) else dict(spec)
    unknown = [v for v in wanted if v not in VERSIONS]
    if unknown:
        print(f"⚠ split spec names {unknown}, which are not in VERSIONS {VERSIONS} — ignored. "
              f"A typo here silently shrinks the run rather than failing.")
    out = {}
    for version in VERSIONS:
        s = wanted.get(version)
        if s is None:
            print(f"[{version}] not named in this run's split spec — dropped from this run.")
            continue
        if s not in config.SPLITS[version]:
            print(f"[{version}] has no split {s!r} (its splits are {config.SPLITS[version]}) "
                  f"— dropped from this run.")
            continue
        out[version] = s
    return out


# WHICH SPLIT EACH RUN READS — the one knob. Per run: the split (a single name, or a
# {version: split} map where the versions disagree) and the out_suffix that names the
# backend's file: "" is the canonical no-suffix parquet (whatever ACTIVE_BACKEND was when it
# was written — interventional), "_native" the sibling 00_SHAP.ipynb §11 writes by appending
# "_native" to attribute.py's --out for the tree-path-dependent backend. The suffix is appended
# AFTER the split, so "_native" is the right spelling on EVERY split
# (v2_attributions_train_native.parquet, v2_attributions_test_native.parquet, …).
#
# Add or delete a line to change what the notebook compares — every section below is driven
# by RESULTS, so nothing downstream needs editing. Run labels reach figure filenames: keep
# them short, and do not rename an existing one without renaming its figures.
RUN_SPEC = {
    "interventional_train": {"splits": "train",          "out_suffix": ""},
    "interventional_oot":   {"splits": config.OOT_SPLIT, "out_suffix": ""},
    "path_dependent":       {"splits": config.OOT_SPLIT, "out_suffix": "_native"},
    # "interventional_val":   {"splits": {"v1": "val1", "v2": "val"}, "out_suffix": ""},
    # "path_dependent_train": {"splits": "train",          "out_suffix": "_native"},
}

RUNS = {name: {"splits": splits_for(cfg["splits"]), "out_suffix": cfg["out_suffix"]}
        for name, cfg in RUN_SPEC.items()}
assert RUNS, "RUN_SPEC is empty — declare at least one run."

# Kept as names because §6 and the decision-side profiles still speak of "the train split"
# and "the holdout" specifically, whatever RUN_SPEC happens to contain.
TRAIN_SPLITS = splits_for("train")
OOT_SPLITS   = splits_for(config.OOT_SPLIT)

for name, cfg in RUNS.items():
    if not cfg["splits"]:
        print(f"[{name}] no version survived its split spec — the run will be empty.")
    else:
        print(f"[{name}] " + ", ".join(f"{v}/{s}" for v, s in cfg["splits"].items())
              + (f"  suffix={cfg['out_suffix']}" if cfg["out_suffix"] else ""))


def _attributions_path(version: str, split: str, out_suffix: str) -> Path:
    """Mirror of the --out-suffix naming convention a caller uses for attribute.py — the one other place that
    convention needs to exist, since a notebook cannot import a CLI driver's local variable."""
    base = config.path("attributions", version, SOURCE, split=split)
    return base.with_name(base.stem + out_suffix + base.suffix) if out_suffix else base


def _read_suffixed(version: str, split: str, out_suffix: str) -> tuple[pd.DataFrame, dict]:
    """Read an attributions parquet + sidecar meta by an EXPLICIT path — needed only for a
    suffixed (non-default-backend) file, which VersionData has no parameter for."""
    p = _attributions_path(version, split, out_suffix)
    meta_p = p.with_name(p.stem + "_meta.json")
    if not p.exists():
        raise FileNotFoundError(
            f"[{version}/{split}{out_suffix}] {p} not found. Produce it with:\n"
            f"    src/envs/{version}/.venv/bin/python src/attribution/attribute.py --split {split} "
            f"--backend native --out {p}\n"
        )
    return pd.read_parquet(p), json.loads(meta_p.read_text(encoding="utf-8"))


## 1 · `analyze()` — the original §1/§2, made callable per run

Everything the notebook did in one pass before is now a function of `(run_label, splits,
out_suffix)`. Figures/prints are tagged with `run_label` so three executions in one kernel don't
overwrite each other's PNGs or read as one blurred comparison.

For a non-empty `out_suffix`, `.attributions`/`.attribution_meta` are read from the suffixed path
and assigned onto the `VersionData` instance BEFORE they are ever touched —
`functools.cached_property` treats a direct assignment exactly like a cache hit (it is a
non-data descriptor: instance-`__dict__` wins), so the normal `attribute.py`-produced file is never
read for that run, and `.frame`/`.decisions` still resolve the ordinary way through the same object.


In [ ]:
def _global_importance_fig(MABS: dict, run_label: str, aliased: bool) -> None:
    """One multi-panel figure, built fresh — not shared with its real/alias twin, so neither
    figure's layout is sized around the other's label widths (see Notes at the bottom)."""
    n = len(MABS)
    fig, axes = plt.subplots(1, n, figsize=(max(figstyle.FIG_1[0], 4.0 * n), 0.30 * TOPN + 1.6))
    for ax, (v, m), colour in zip(np.atleast_1d(axes), MABS.items(), figstyle.SERIES):
        top = m.head(TOPN)[::-1]
        labels = feature_alias.to_alias(v, top.index) if aliased else list(top.index)
        bars = ax.barh(np.arange(len(top)), top.values, color=colour)
        ax.set_yticks(np.arange(len(top)))
        ax.set_yticklabels(labels, fontsize=7)
        ax.set_title(f"{v} — top {TOPN} of {len(m)}")
        ax.set_xlabel("mean |φ|  (log-odds)")
        ax.bar_label(bars, fmt="%.3f", fontsize=6, padding=2)
        ax.margins(x=0.12)     # headroom so the value label never clips past the axes edge
    fig.suptitle(f"Attribution mass by feature — {run_label}" + (" (aliased)" if aliased else ""))
    fig.tight_layout()
    prefix = "alias_" if aliased else ""
    figstyle.save_fig(fig, f"{prefix}01a_real_shap_global_importance_{run_label}")
    plt.show()


def analyze(run_label: str, splits: dict[str, str], out_suffix: str = "") -> dict:
    avail, missing = {}, {}

    # laod files using Versiondata Class 
    for v in splits:                    # the run's OWN versions — splits_for may drop one
        d = load(v, SOURCE, split=splits[v])
        try:
            if out_suffix:
                phi, meta = _read_suffixed(v, splits[v], out_suffix)
                d.attributions = phi              # pre-seeds the cached_property — see note above
                d.attribution_meta = meta
            else:
                d.attributions                     # forces the default read; raises if absent
            avail[v] = d
        except FileNotFoundError as exc:
            missing[v] = str(exc).replace("\n", " ")

    # report header 
    print(f"\n=== {run_label}  ({', '.join(f'{v}/{s}' for v, s in splits.items())}"
          f"{out_suffix} ===")
    for v, why in missing.items():
        print(f"[{v}] NOT attributed yet — {why}\n")
    if not avail:
        print(f"[{run_label}] no version has attributions on disk — skipping.")
        return {"avail": {}, "metas": {}, "MABS": {}}

    # check required key fields for feature concentration analysis 
    metas = {v: d.attribution_meta for v, d in avail.items()}
    conc.require_comparable(metas)         # raises on mixed backends WITHIN this run

    FIELDS = ("split", "backend", "perturbation", "model_output", "estimator", "feature_order",
              "n_rows", "n_features", "background_n", "base_value")
    display(pd.DataFrame({v: {k: m.get(k) for k in FIELDS} for v, m in metas.items()}).T)

    MABS = {v: d.mean_abs_shap for v, d in avail.items()}     # mean|φ| per feature, descending

    # TWO INDEPENDENT figures, real name and alias — each its own plt.subplots()/tight_layout()/
    # savefig(). The real-name PNG is company-laptop/internal only; alias_ is the only version
    # safe to leave the machine.
    _global_importance_fig(MABS, run_label, aliased=False)
    _global_importance_fig(MABS, run_label, aliased=True)

    return {"avail": avail, "metas": metas, "MABS": MABS}


RESULTS = {name: analyze(name, cfg["splits"], cfg["out_suffix"]) for name, cfg in RUNS.items()}

assert any(res["avail"] for res in RESULTS.values()), (
    "no run produced any attributions. Run attribute.py for each version, then re-run this"
    "notebook.")


## 2 · Feature mapping — computed once, not per run

A version's trained feature *columns* don't change with which split or backend attributed them —
only the φ magnitudes do. So the hand-confirmed mapping (originally §3) is resolved once, against
the union of whichever versions any run managed to load, and reused by every run's concentration
tables below.

**One basis per version SUBSET — including each version on its own.** A *basis* is the feature set
the shares p_i are normalised over; two numbers compare only when they sit on the **same** basis.
`BASES` holds one per subset of the loaded versions:

| basis | feature set | what it is for |
|---|---|---|
| `(own features)` — 1-way, one per version | that version's FULL encoded column set | the version's individual concentration; D0 = its column count, so only the evenness ratios travel across versions |
| `v2+v3` etc. — pairs | mapped rows present in BOTH | the direct D1/D2 comparison for that pair — `v2+v3` (52 mapped features on the real data) is the pair the SFP claim rests on |
| `v1+v2+v3` — all loaded versions | mapped rows present in ALL | the strictest set; only 9 features on the real data |

An intersection is a *minimum*: a version that shares few features starves the comparison for the
other two. That is the shape the real mapping actually has — 9 features span all three versions,
52 pair v2–v3 — so restricting to the three-way set alone would discard exactly the v2→v3 evidence
the SFP argument rests on, to keep a version that is not in the claim. Every section below (§3,
§4, §5, §6) therefore iterates over `BASES`, and §3c audits the zero-mass columns that make
`richness_D0` fall short of a basis's column count.

In [ ]:
ALL_MABS = {}
for res in RESULTS.values():
    ALL_MABS.update(res["MABS"])   # a version's raw column set is the SAME regardless of which
                                   # run attributed it — later runs just confirm the same index

names = {v: set(m.index) for v, m in ALL_MABS.items()}

# ── (a0) cross-check against the pickle-extracted registry ────────────────────────────
# The φ columns arrive from the attributions parquet, which attribute.py wrote from the booster's
# own feature list. Nothing downstream re-checks them, so a parquet written against a different
# matrix — a re-export, a changed feature set, a stale file — would flow straight into the
# concentration numbers looking perfectly healthy. features/registry/<v>.json is the independent
# record of what each model consumes (extract_features.py, run inside that version's env), so
# comparing the two costs one read and catches exactly that.
for v, have in sorted(names.items()):
    reg_path = config.registry_path(v)
    if not reg_path.exists():
        print(f"[{v}] no registry ({reg_path.name}) — φ column set UNVERIFIED. "
              f"Run features/extract_features.py inside env-{v}.")
        continue
    known = {str(c) for c in (json.loads(reg_path.read_text(encoding="utf-8"))
                              .get("model_features") or [])}
    if not known:
        print(f"[{v}] registry lists no model_features — φ column set UNVERIFIED.")
        continue
    only_phi, only_reg = sorted(have - known), sorted(known - have)
    if only_phi or only_reg:
        print(f"⚠ [{v}] φ columns disagree with {reg_path.name}: "
              f"{len(only_phi)} only in φ {only_phi[:5]}, "
              f"{len(only_reg)} only in registry {only_reg[:5]} — "
              f"re-run attribute.py for {v}, or rebuild the registry; do not report these numbers.")
    else:
        print(f"[{v}] φ columns confirmed against {reg_path.name} ({len(have)} features)")
print()


In [ ]:
# ── (a) raw string-equality intersection — the encoding-divergence EXHIBIT, not the basis
RAW_INTER = set.intersection(*names.values()) if len(names) > 1 else set(next(iter(names.values())))
print(f"raw string intersection across {list(names)}: {len(RAW_INTER)} features")
print("  (expected to be small — same concepts, different encodings. Comparison uses the")
print("   hand-confirmed mapping below, never name equality.)\n")

# ── (b) the hand-confirmed mapping: {"<idx>": {"v1": name, "v2": name, "v3": name}, ...}
overlap_json = ROOT / "features" / "feature_overlap.json"
MAPPING = {}     # idx -> {version: that version's own encoded feature name}
have = list(ALL_MABS)
if overlap_json.exists():
    MAPPING = {int(k): row for k, row in json.loads(overlap_json.read_text()).items()}
    full_rows = {k: row for k, row in MAPPING.items()
                 if all(v in row and row[v] in names[v] for v in have)}
    dropped = [k for k, row in MAPPING.items()
               if all(v in row for v in have) and k not in full_rows]
    if dropped:
        print(f"⚠ {len(dropped)} mapped rows name features ABSENT from the attributions "
              f"{dropped[:8]}{'…' if len(dropped) > 8 else ''} — stale mapping or wrong matrix; "
              f"resolve before reporting.")
    print(f"hand-confirmed mapping: {len(MAPPING)} entries; {len(full_rows)} usable across "
          f"{have} and present in the attributions")
    for v in have:
        mapped_v = {row[v] for row in full_rows.values()}
        held = (float(ALL_MABS[v][ALL_MABS[v].index.isin(mapped_v)].sum() / ALL_MABS[v].sum())
                if mapped_v else 0.0)
        print(f"  {v}: {len(names[v]):>4} encoded · {len(mapped_v):>3} mapped · "
              f"mapped set holds {held:.1%} of its attribution mass")
    if len(have) > 1:
        pairs = pd.DataFrame(
            [[sum(1 for row in MAPPING.values() if a in row and b in row) for b in have]
             for a in have], index=have, columns=have)
        print("\nmapped features per version pair (diagonal = that version's mapped total):")
        display(pairs)
        # Index/columns are VERSION labels (v1/v2/v3), not Allianz feature names -- plain export,
        # no alias twin needed. See the Notes cell at the bottom.
        figstyle.save_table(pairs, "feature_overlap_pairs_per_version")
else:
    print(f"{overlap_json} not built yet — run features/check_overlap.py (needs the Excel; "
          f"company laptop).\nNo mapped basis can be built: BASES below holds only the 1-way "
          f"(own features) bases, so no restricted number exists to report.")

# ── (c) one comparison BASIS per version subset — 1-way included ──────────────────────
# A basis is the feature set the shares are normalised over; two numbers compare only on the
# SAME basis. The mapped intersection across EVERY loaded version is a MIN: one version that
# shares little starves the comparison for the other two. The real mapping is exactly that shape
# — 9 features span all three, 52 pair v2–v3 — and v2→v3 is the pair the SFP argument rests on
# (problem.md), so an all-versions-only restriction would throw the main evidence away to keep a
# version the claim does not involve. Each subset therefore gets its own basis, and `basis` is an
# index level in §5 rather than a footnote.
#
# Size-1 subsets are the INDIVIDUAL result — each version on its FULL column set, not on the
# mapped-anywhere subset — so D0 there is the version's column count and only the evenness
# ratios travel across versions. Reported as basis "(own features)" in §3 / §4 / §5 / §6, and
# audited for zero-mass columns in §3c.
BASIS_MIN = 5       # a MAPPED basis narrower than this is printed but never compared
OWN = "(own features)"


def basis_label(subset: tuple[str, ...]) -> str:
    """Index label / figure title for a basis. All 1-way bases share the one label."""
    return OWN if len(subset) == 1 else "+".join(subset)


def basis_slug(subset: tuple[str, ...]) -> str:
    """Filename form of basis_label."""
    return "own" if len(subset) == 1 else "".join(subset)


def basis_n(basis: dict[str, set]) -> int:
    """Feature count of a basis — equal across its versions by construction for a mapped one."""
    return min((len(x) for x in basis.values()), default=0)


def _basis(subset: tuple[str, ...]) -> dict[str, set]:
    if len(subset) == 1:                              # 1-way: the version's own full column set
        return {subset[0]: set(names[subset[0]])}
    rows = [row for row in MAPPING.values()
            if all(v in row and row[v] in names[v] for v in subset)]
    return {v: {row[v] for row in rows} for v in subset}


BASES: dict[tuple[str, ...], dict[str, set]] = {}
for _size in range(len(have), 0, -1):                 # all-versions, then pairs, then 1-way
    for _subset in itertools.combinations(have, _size):
        _b = _basis(_subset)
        if basis_n(_b):
            BASES[_subset] = _b

MAPPED_BASES = {s: b for s, b in BASES.items() if len(s) > 1}
if not MAPPED_BASES:
    print("\nno mapped basis — every restricted section below is empty; only (own features) runs.")

BASES_SUMMARY = pd.DataFrame([
    {"basis": basis_label(sub) if len(sub) > 1 else f"{OWN} {sub[0]}",
     "features": basis_n(b),
     **{f"mass_{v}": (float(ALL_MABS[v][ALL_MABS[v].index.isin(b[v])].sum()
                            / ALL_MABS[v].sum()) if v in b else np.nan) for v in have}}
    for sub, b in BASES.items()]).set_index("basis")
print("\ncomparison bases — feature count, and the share of each version's attribution mass the\n"
      "basis holds (1.0 on a version's own basis by definition):")
display(BASES_SUMMARY.round(3))
# Index is a basis label ("(own features) v2", "v2+v3", ...), not a feature name -- plain export.
figstyle.save_table(BASES_SUMMARY, "comparison_bases_summary")

_all = tuple(have)
if len(have) > 1 and _all in BASES:
    n_all = basis_n(BASES[_all])
    wider = [(basis_label(sub), basis_n(b)) for sub, b in MAPPED_BASES.items()
             if len(sub) < len(have) and basis_n(b) > n_all]
    if wider:
        print(f"\n⚠ a PAIR basis is wider than the {len(have)}-way one ({n_all} features): "
              + ", ".join(f"{k} has {n}" for k, n in wider)
              + f"\n  The {len(have)}-way intersection is a min, so reporting only it drops "
                f"features the pair DOES share.\n  Report the pair the claim rests on, on its own "
                f"basis, and NAME the basis beside every number.")

## 3 · Concentration, per run × per basis

One table per run per basis in `BASES` (§2): the `(own features)` table first — every loaded
version on its full column set, i.e. the 1-way / individual result — then one per mapped basis.
Where each number is reported again: the same profiles reappear as rows of the §5 cross-run table
(index level `basis`), the Hill curves behind them are §4's figures (one per basis), and the
zero-mass columns that make `richness_D0` fall short of `n_cols` are listed in §3c.

Direction of the hypothesis, unchanged: if the loop is concentrating the model's reasoning onto the
fast-track drivers, later versions should show **lower** D1/D2 and **higher** Simpson / Gini /
top-k share. Compare row-by-row **within a version**: `interventional_train` vs
`interventional_oot` is the temporal question (does the fitted function's concentration generalise,
or is train-only overfitting driving it); `interventional_oot` vs `path_dependent` is the backend
question (is the concentration finding an artefact of the reference distribution). §5 turns this
into one table.

In [ ]:
# PROFILES[run][basis_label] -> one profile row per version on that basis.
# Where each result is reported:
#   (own features)  — the 1-way / individual result: this table; §5 rows basis="(own features)";
#                     §4 figure "..._own"; §3c (which of its columns are exactly zero); §6 per side.
#   mapped bases    — this table; §5 rows on the same basis label; §4 figure "..._<basis>";
#                     §6 per side. v2+v3 is the basis the thesis claim reads.
PROFILES = {}
for name, res in RESULTS.items():
    if not res["MABS"]:
        continue
    PROFILES[name] = {}

    own = {s[0]: res["MABS"][s[0]] for s in BASES if len(s) == 1 and s[0] in res["MABS"]}
    if own:
        table = conc.profile_table(own, k=TOPK)
        table.insert(0, "n_cols", [len(m) for m in own.values()])
        print(f"\n[{name}] {OWN} — each version on its full column set. richness_D0 < n_cols "
              f"means zero-mass columns (§3c); D0 differs by construction, compare evenness only.")
        display(table.round(4))
        PROFILES[name][OWN] = table

    for subset, basis in MAPPED_BASES.items():
        if any(v not in res["MABS"] for v in subset):
            continue                          # a basis needs every one of its versions in this run
        label, n = basis_label(subset), basis_n(basis)
        if n < BASIS_MIN:
            print(f"[{name}] {label}: only {n} mapped features — too thin for a comparison; skipped.")
            continue
        restricted = {v: res["MABS"][v][res["MABS"][v].index.isin(basis[v])] for v in subset}
        table = conc.profile_table(restricted, k=TOPK)
        table.insert(0, "n_cols", n)
        print(f"[{name}] {label} — {n} mapped features; D1/D2 directly comparable within this basis.")
        display(table.round(4))
        PROFILES[name][label] = table

# Every (run, basis) profile table above, full CSV. Index is a VERSION, not a feature name (these
# are concentration measures, not per-feature values) -- plain export, no alias twin needed.
for _run_name, _by_basis in PROFILES.items():
    for _label, _table in _by_basis.items():
        _slug = _label.replace(" ", "_").replace("(", "").replace(")", "")
        figstyle.save_table(_table, f"profile_{_run_name}_{_slug}")

### 3b · The configuration that produced those numbers — also computed once

`problem.md` §1.4c: **no adjacent version pair shares a configuration.** This is a property of the
fitted estimator, not of which split or backend attributed it, so — unlike §3 — there is exactly
one params table, not three.


In [ ]:
ALL_METAS = {}
for res in RESULTS.values():
    ALL_METAS.update(res["metas"])   # same fitted model regardless of split/backend, so the
                                     # params agree across runs; last write is as good as any

KNOBS = ("n_estimators", "max_depth", "max_leaves", "min_child_weight", "learning_rate", "eta",
         "reg_alpha", "reg_lambda", "gamma", "subsample", "colsample_bytree",
         "scale_pos_weight", "eval_metric", "objective")

params = {v: m.get("estimator_params") or {} for v, m in ALL_METAS.items()}
if any(params.values()):
    cfg = pd.DataFrame({v: {k: p.get(k, "—") for k in KNOBS} for v, p in params.items()})
    # column count beside the regularisation knobs — §3c reads the two together
    cfg.loc["n_features"] = [ALL_METAS[v].get("n_features", "—") for v in cfg.columns]
    cfg = cfg.reindex(["n_features", *KNOBS])
    display(cfg)
    # Index is an xgboost HYPERPARAMETER name (n_estimators, max_depth, ...), never an Allianz
    # feature name -- plain export, no alias twin needed.
    figstyle.save_table(cfg, "hyperparameter_comparison")
    if len(params) > 1:
        differs = [k for k in KNOBS if len({str(p.get(k)) for p in params.values()}) > 1]
        if differs:
            print(f"knobs that DIFFER: {differs}\n"
                  "The concentration difference above is confounded with these — report it as "
                  "*consistent with* the loop, not as identifying it, unless a matched-"
                  "hyperparameter refit or a sensitivity sweep bounds them (problem.md §1.4c i–iii).")
        else:
            print("these versions are configuration-matched on the knobs above — the "
                  "regularisation confound of problem.md §1.4c does not apply to this pair.")
else:
    print("no estimator params in the meta — re-run attribute.py for that version to capture them "
          "(they can only be read where the pickle opens).")

### 3c · Zero-mass columns — is `richness_D0` short of the column count, and where?

The two live versions are regularised very differently: v2 (77 columns) with `reg_alpha=20`,
`gamma=15`, `reg_lambda≈0.012`; v3 (162 columns) with `reg_alpha=0`, `gamma≈0.0005`,
`reg_lambda≈1.18`. `gamma` prunes any split whose gain falls below it and `reg_alpha` shrinks leaf
weights towards zero, so v2's booster is expected to leave columns it never splits on — and a
column no tree reads gets mean|φ| = **exactly 0** under both backends. Those columns are what
makes `richness_D0` fall below `n_cols` (`conc.shares` drops zero mass before any measure).

This matters for the concentration story because a zero column is a regularisation effect, not
evidence of the loop concentrating the model's reasoning: the same v2-vs-v3 gap in D0 would
appear with the same regularisation and no forced labels at all (`problem.md` §1.4c). So the audit
runs per version × run — the exact-zero set, a softer "negligible" bucket, the knobs beside them —
and once more inside every mapped basis, where a shared feature that v2 zeroes and v3 uses is
the case the §5 `v2+v3` rows have to be read against. v1 and v3 get the same treatment.

In [ ]:
# ── §3c · zero-mass columns, per version × run ───────────────────────────────────────
# Where it is reported: THIS table and the name list only. §3/§5 already absorb it silently —
# richness_D0 is the count of columns with mean|φ| > 0 (conc.shares drops zero mass), so
# n_cols − richness_D0 there equals n_zero here. §5 carries n_cols next to richness_D0 for that.
NEAR_SHARE = 1e-3     # second, softer bucket: share of total mass below which a column is negligible

rows, ZERO_COLS = {}, {}
for name, res in RESULTS.items():
    for v, m in res["MABS"].items():
        p = m / m.sum()
        zero = list(m.index[m == 0])
        near = m.index[(m > 0) & (p < NEAR_SHARE)]
        par = ALL_METAS.get(v, {}).get("estimator_params") or {}
        rows[(v, name)] = {
            "n_cols": len(m),
            "richness_D0": int((m > 0).sum()),
            "n_zero": len(zero),
            f"n_negligible(share<{NEAR_SHARE:g})": int(len(near)),
            "mass_negligible": float(p.loc[near].sum()),
            **{k: par.get(k, "—") for k in ("reg_alpha", "reg_lambda", "gamma", "min_child_weight")},
        }
        ZERO_COLS[(v, name)] = zero

zero_table = pd.DataFrame(rows).T
zero_table.index.names = ["version", "run"]
display(zero_table.sort_index())
# Index is (version, run), not a feature name -- plain export, no alias twin needed.
figstyle.save_table(zero_table.sort_index(), "zero_mass_columns_summary")

# A zero column is a property of the fitted trees — never split on, or split on only where the
# leaf contribution is nil — so the SET must agree across runs of the same version (split and
# backend change the rows and the reference, not the trees). If it does not, the run's row sample
# or background is producing the zero, not the fit.
for v in sorted({v for v, _ in ZERO_COLS}):
    sets = {run: frozenset(z) for (vv, run), z in ZERO_COLS.items() if vv == v}
    if len(set(sets.values())) > 1:
        print(f"⚠ [{v}] zero-mass set differs across runs: "
              + ", ".join(f"{run}={len(z)}" for run, z in sets.items())
              + " — a sample effect, not a property of the fit; do not read n_zero as 'unused'.")

for (v, name), z in ZERO_COLS.items():
    if z:
        print(f"\n[{v}/{name}] {len(z)} column(s) with mean|φ| exactly 0:")
        print("   " + ", ".join(z[:30]) + (" …" if len(z) > 30 else ""))

# The zero-mass COLUMN NAMES themselves, real name then alias — this is the one place in this
# notebook where actual Allianz feature names sit in a table rather than an aggregate concentration
# measure (the print above only ever showed the first 30). Long format: one row per
# (version, run, feature).
ZERO_COLS_LONG = pd.DataFrame(
    [{"version": v, "run": run, "feature": f} for (v, run), z in ZERO_COLS.items() for f in z])
figstyle.save_table(ZERO_COLS_LONG, "zero_mass_columns_list")
if not ZERO_COLS_LONG.empty:
    _parts = []
    for v, grp in ZERO_COLS_LONG.groupby("version"):
        grp = grp.copy()
        grp["feature"] = feature_alias.to_alias(v, list(grp["feature"]))
        _parts.append(grp)
    figstyle.save_table(pd.concat(_parts, ignore_index=True), "alias_zero_mass_columns_list")

# The same count inside each mapped basis: a shared feature one version zeroes and the other uses
# is where regularisation, not the loop, moves the restricted D0. n_cols − richness_D0 in §5 on
# that basis is this number.
if MAPPED_BASES:
    inb = {}
    for subset, basis in MAPPED_BASES.items():
        for name, res in RESULTS.items():
            for v in subset:
                if v in res["MABS"]:
                    m = res["MABS"][v]
                    inb[(basis_label(subset), v, name)] = {
                        "n_in_basis": len(basis[v]),
                        "n_zero_in_basis": int((m[m.index.isin(basis[v])] == 0).sum())}
    zero_in_basis = pd.DataFrame(inb).T
    zero_in_basis.index.names = ["basis", "version", "run"]
    print("\nzero-mass columns inside each mapped basis:")
    display(zero_in_basis.sort_index())
    # Index is (basis, version, run) -- plain export, no alias twin needed.
    figstyle.save_table(zero_in_basis.sort_index(), "zero_mass_columns_in_basis")

## 4 · Diversity profile — one figure per basis, one panel per version, the three runs overlaid

Each figure is one basis from `BASES` (§2). The `(own features)` figure shows every version on its
full column set — the individual profile, whose D0 is that version's column count — and each
mapped basis shows only the versions it spans, restricted to the mapped set, so panels within one
figure sit on the same feature set and are directly comparable. Curves diverging at small q (the
tail) vs large q (the dominant features) say different things; the scalar D1/D2 numbers in §3 are a
summary of exactly this picture. Filenames `04<letter>_real_shap_hill_profile_by_run_<basis>`,
letters fixed by basis order (all-versions, pairs, own).

In [ ]:
# q values worth reading off directly: 0=richness D0, 1=exp-Shannon D1, 2=inverse-Simpson D2.
HILL_Q_MARKS = (0.0, 1.0, 2.0)
# Runs are often close in SHAPE (same underlying feature set), so colour alone does not separate
# their curves — pair each run with its own linestyle/marker too, cycling if there are ever more
# runs than styles.
_RUN_STYLES = ["-", "--", ":", "-."]
_RUN_MARKERS = ["o", "s", "^", "D"]
_LABEL_OFFSETS = (10, -18, 24, -30)   # points, alternating above/below so per-run value labels at
                                      # the same (q, D_q) don't stack on top of each other


def hill_figure(title: str, sets: dict[str, set], fname: str) -> None:
    """One panel per version in `sets`, one curve per run; every panel on the same basis.

    Each run gets its own colour + linestyle + marker, and its D_q at HILL_Q_MARKS is written
    next to the curve — without that, runs whose diversity profiles are close in shape are
    indistinguishable at a glance, especially on the log y-axis.
    """
    versions = list(sets)
    fig, axes = plt.subplots(1, len(versions), figsize=(4.2 * len(versions), 3.6), squeeze=False)
    for ax, v in zip(axes[0], versions):
        for i, ((name, res), colour) in enumerate(zip(RESULTS.items(), figstyle.SERIES)):
            if v not in res["MABS"]:
                continue
            m = res["MABS"][v]
            curve = conc.hill_curve(m[m.index.isin(sets[v])])
            style = _RUN_STYLES[i % len(_RUN_STYLES)]
            marker = _RUN_MARKERS[i % len(_RUN_MARKERS)]
            mark_idx = sorted({int(np.argmin(np.abs(curve.index.values - q)))
                                for q in HILL_Q_MARKS})
            ax.plot(curve.index, curve.values, color=colour, linestyle=style, marker=marker,
                    markevery=mark_idx, markersize=5, linewidth=1.6, label=name)
            offset = _LABEL_OFFSETS[i % len(_LABEL_OFFSETS)]
            for idx in mark_idx:
                q, val = curve.index[idx], curve.values[idx]
                ax.annotate(f"{val:.1f}" if val >= 10 else f"{val:.2f}", (q, val),
                            textcoords="offset points", xytext=(0, offset), fontsize=6.5,
                            color=colour, ha="center", va="bottom" if offset > 0 else "top")
        ax.set_title(f"{v}  ({len(sets[v])} features)")
        ax.set_xlabel("order  q")
        ax.set_yscale("log")
        ax.margins(y=0.18)     # headroom for the value labels above/below the topmost curve
    axes[0][0].set_ylabel("effective number of features  Dq")
    axes[0][-1].legend(title="run", fontsize=7)
    fig.suptitle(f"Attribution diversity profile — {title}")
    fig.tight_layout()
    figstyle.save_fig(fig, fname)
    plt.show()


# One figure per basis. Where it is reported: "04<letter>_real_shap_hill_profile_by_run_<basis>"
# — the "(own features)" figure is the individual (1-way) profile, each mapped basis its own
# figure. Letters follow BASES order (all-versions, pairs, then own) and are assigned BEFORE thin
# bases are skipped, so a skipped basis leaves a gap rather than renaming the rest.
_specs = [(basis_label(s), b, basis_slug(s)) for s, b in MAPPED_BASES.items()]
_own_sets = {s[0]: b[s[0]] for s, b in BASES.items() if len(s) == 1}
if _own_sets:
    _specs.append((OWN, _own_sets, "own"))

for letter, (label, sets, slug) in zip("abcdefghijklmnopqrstuvwxyz", _specs):
    if label != OWN and basis_n(sets) < BASIS_MIN:
        print(f"[§4] {label}: only {basis_n(sets)} mapped features — figure skipped.")
        continue
    title = label if label == OWN else f"{label} ({basis_n(sets)} mapped features)"
    hill_figure(title, sets, f"04{letter}_real_shap_hill_profile_by_run_{slug}")

## 5 · Cross-run comparison — one table, the direct answer to "how does it differ"

`evenness_D2_D0` (dominance-weighted, unit-free) is the single number the SFP direction hypothesis
rests on — lower under the loop hypothesis. The table carries three index levels and all three have
to be read:

* **basis** — which feature set the shares were computed over. `(own features)` is each version's
  full column set (the 1-way basis of §2), where D0 differs by construction, so only the evenness
  ratios mean anything across versions. Every other row is one of §2's mapped subsets, where D1/D2
  are directly comparable *within that basis*. A v2 number on the `v2+v3` basis and a v2 number on
  the `v1+v2+v3` basis are two different statistics — never read the gap between them as a change.
  `n_cols` is the basis size and `richness_D0` how many of those columns carry any mass; the gap
  between them is §3c's zero-mass set.
* **version** — reading DOWN a version's runs on ONE basis: `train → oot` should look similar if
  the loop is a property of the fitted function rather than of the particular split; a large jump
  questions the finding, not the split. `interventional_oot → path_dependent` is the backend-only
  move, taken apart feature by feature in §7.
* **run** — the configuration from §0.

The row that carries the thesis claim is the v2 → v3 pair on the **`v2+v3` basis**, because that
basis keeps every feature those two share (52) instead of only the ones v1 also happens to have (9).

In [ ]:
COLS = ("richness_D0", "evenness_D1_D0", "evenness_D2_D0", "simpson_S", "gini", f"top{TOPK}_share")

# Every basis in BASES, 1-way included: rows labelled "(own features)" are the individual result,
# every other label a mapped subset. A basis needs every one of its versions in the run.
rows = {}
for name, res in RESULTS.items():
    for subset, basis in BASES.items():
        if any(v not in res["MABS"] for v in subset):
            continue
        if len(subset) > 1 and basis_n(basis) < BASIS_MIN:
            continue
        for v in subset:
            m = res["MABS"][v]
            m = m[m.index.isin(basis[v])]
            prof = conc.profile(m, k=TOPK)
            rows[(basis_label(subset), v, name)] = {"n_cols": len(m),
                                                    **{c: prof[c] for c in COLS}}

comparison = pd.DataFrame(rows).T
comparison.index.names = ["basis", "version", "run"]
comparison = comparison.astype({"n_cols": int, "richness_D0": int})
display(comparison.round(4).sort_index())
# Index is (basis, version, run) -- plain export, no alias twin needed.
figstyle.save_table(comparison.sort_index(), "cross_run_concentration_comparison")

## 6 · Concentration either side of the fast-track cutoff

The SFP mechanism only operates on claims that were **scrapped** — those are the rows whose label
was forced. So the sharper version of the question is whether concentration differs between the
scrapped and the garage-assessed side of that version's own rule (v1 segmented on mobility, v2
piecewise in time, v3 global — `threshold.apply` dispatches; nothing here assumes a single τ).

This is descriptive, not causal: the two sides differ in case-mix as well as in treatment. It is
the input to the estimator layer, not a result on its own.

Decisions are reproduced once per version, then profiled on every basis in `BASES`, so the table
carries `(basis, version, side)`. v1 never appears — its rule needs the production log, which is
gone — so the rows to read are v2 and v3 on the `v2+v3` basis; the `v1+v2+v3` rows restrict those
same two versions to the 9 three-way features for no gain and are kept only so every basis is
present. Shown for `interventional_oot` only, matching the original default —
`decision_side_profile()` takes any run label, so the other two are one call away.

In [ ]:
def decision_side_mabs(run_label: str) -> dict:
    """mean|φ| per (version, side) for one run. Decisions are reproduced ONCE per version here;
    the basis restriction happens afterwards, so every basis reads the same split of rows."""
    res = RESULTS.get(run_label, {})
    out = {}
    for v, d in res.get("avail", {}).items():
        # PER-VERSION guard, not one try around the whole loop. `d.decisions` reproduces that
        # version's own rule and needs artefacts that do not all exist: v1's rule is segmented on
        # mobility and `VersionData.decisions` reads it from the production LOG — which v1 does
        # not have and never will (destroyed; `paths.log_source = None`). A single try/except
        # around the loop let v1's FileNotFoundError discard v2's and v3's rows too, turning a
        # known-missing artefact into an empty table.
        try:
            frame = d.frame[[schema.CLAIM_ID]].copy()
            frame["decision"] = d.decisions
        except (FileNotFoundError, KeyError, ValueError) as exc:
            print(f"[{run_label}/{v}] decisions cannot be reproduced — {str(exc).splitlines()[0]}")
            continue

        att = d.attributions.merge(frame, on=schema.CLAIM_ID, how="inner")
        if att.empty:
            print(f"[{run_label}/{v}] no attributed claim appears in the scored frame — skipping")
            continue
        for side, sub in att.groupby("decision"):
            label = "scrapped" if side == 1 else "garage"
            if len(sub) < 50:
                print(f"[{run_label}/{v}] {label}: only {len(sub)} rows — too thin, skipping")
                continue
            out[(v, label)] = (conc.mean_abs(sub.drop(columns=["decision"]), id_col=schema.CLAIM_ID),
                               len(sub))
    return out


def decision_side_profile(run_label: str):
    """Concentration either side of the cutoff on every basis in BASES — index (basis, version, side).

    Where it is reported: this table only. Read scrapped vs garage WITHIN a version on ONE basis.
    "(own features)" rows are the individual result; v2 and v3 on the v2+v3 basis are the rows
    the claim reads.
    """
    sides = decision_side_mabs(run_label)
    if not sides:
        print(f"[{run_label}] no version produced a usable decision-side profile.")
        return None

    rows = {}
    for subset, basis in BASES.items():
        if len(subset) > 1 and basis_n(basis) < BASIS_MIN:
            continue
        for (v, side), (m, n) in sides.items():
            if v not in subset:
                continue
            mm = m[m.index.isin(basis[v])]
            if mm.empty or float(mm.sum()) <= 0:
                print(f"[{run_label}/{v}] {side} on {basis_label(subset)}: no attribution mass — skipping")
                continue
            rows[(basis_label(subset), v, side)] = {"n_rows": n, "n_cols": len(mm),
                                                    **conc.profile(mm, k=TOPK)}

    by_side = pd.DataFrame(rows).T
    by_side.index.names = ["basis", "version", "side"]
    by_side = by_side.astype({"n_rows": int, "n_cols": int, "richness_D0": int})
    display(by_side.round(4).sort_index())
    # Index is (basis, version, side) -- plain export, no alias twin needed. run_label in the
    # filename so a future train/path_dependent call (commented out below) does not overwrite this.
    figstyle.save_table(by_side.sort_index(), f"decision_side_profile_{run_label}")
    return by_side


by_side_oot = decision_side_profile("interventional_oot")
# by_side_train    = decision_side_profile("interventional_train")   # uncomment to compare
# by_side_pathdep  = decision_side_profile("path_dependent")

## 7 · The two backends, feature by feature — what actually changes when the reference changes

§5 says *how much* `interventional_oot` and `path_dependent` differ. This says *where*. It is a
**within-version** comparison, so no cross-version mapping is involved: each version is compared
against itself, on the same split, the same rows, and its own full feature set. Backend is the only
variable.

**What the two backends are** — worth stating precisely, because the loose version is wrong. Both
explain the *same* predictions of the *same* fitted model. They differ in the reference
distribution used for "this feature is absent":

* **`interventional`** (`shap`, with a background sample) replaces an absent feature with draws
  from the background **independently of the present ones**, so it evaluates the model at input
  combinations that may never occur in the data. It is *true to the model*: φ tracks the fitted
  function, and a feature the function does not read gets exactly 0 no matter what it correlates
  with. Prefer it here — the SFP claim is about the fitted function concentrating, so the
  reference should not smuggle a data distribution in.
* **`tree_path_dependent`** (each booster's own TreeSHAP, no background) walks the trees and
  weights branches by the **cover counts stored in them** — the training distribution, as the tree
  recorded it. It stays on the data manifold, and the reference is *a different dataset per
  version*, which makes any cross-version difference under it partly a comparison of training
  distributions rather than of fitted functions.

So it is not "what it learned" versus "what it predicts" — both explain the same predictions. The
difference is *whose distribution answers the counterfactual*.

⚠️ **`delta_share` has no predictable sign, and reading one into it is a mistake.** Two effects
push opposite ways, and which dominates is a property of the geometry, not of the feature:
interventional withholds the credit a proxy was earning only through correlation (pushing its share
*down*), but it also fires that proxy's splits on off-manifold rows the data never produces
(pushing its share *up*). A worked case: with `x2 = 0.95·x1` and a label depending only on `x1`,
the trees still split on `x2`, and interventional ends up giving `x2` *more* share than
path-dependent, not less. What the column identifies reliably is **which** features are
reference-sensitive — the structurally entangled ones — never a verdict on which is "really" used.
One case is unambiguous: a feature absent from every tree scores exactly 0 under both.

| column | says |
|---|---|
| `delta_share` | share under interventional minus share under path-dependent. Large **magnitude** = this feature's credit depends on the reference, i.e. it is entangled with others. The **sign** is not interpretable on its own (see the warning above) — pair it with `row_corr` before saying anything about it. |
| `rank_move` | places climbed going from path-dependent to interventional. |
| `row_corr` | per-claim agreement of the two φ columns. High, alongside a large `delta_share`, means the backends order claims the same way but disagree on magnitude — a scale disagreement, not a different story. Low means they disagree claim by claim, which is the more serious case. |
| `TV_distance` | ½·Σ&#124;Δshare&#124;, one number in [0, 1] for how far the whole profile moved. |
| `spearman_rank` | whether the feature *ordering* survives the swap at all. |
| `d_evenness` | `evenness_D2_D0` interventional minus path-dependent. The one that matters for the thesis. |

**What would invalidate §3–§5:** `d_evenness` differing in *sign* across versions, or a
`TV_distance` large enough that the two backends put different features at the top. Either means
the concentration result is a statement about the reference distribution, not about the model — so
this table belongs beside the headline number, not in an appendix.


In [ ]:
def compare_backends(run_int: str, run_tpd: str, top: int = 6):
    """Per-feature backend comparison, one version at a time. Same split, same rows, own features.

    The two guards are the substance, not boilerplate. Comparing runs on DIFFERENT splits would
    make this a split comparison wearing a backend label — the exact confound §0 avoids by
    building `path_dependent` on `interventional_oot`'s split. And comparing two runs that turn
    out to share a perturbation measures nothing at all, while looking completely healthy.
    """
    A, B = RESULTS.get(run_int, {}), RESULTS.get(run_tpd, {})
    if not A.get("avail") or not B.get("avail"):
        print(f"[{run_int}] or [{run_tpd}] loaded nothing — skipping.")
        return None, None

    per_feature, summary = {}, {}
    for v in sorted(set(A["avail"]) & set(B["avail"])):
        ma, mb = A["metas"][v], B["metas"][v]
        if ma.get("split") != mb.get("split"):
            print(f"[{v}] {run_int} is on {ma.get('split')!r} and {run_tpd} on "
                  f"{mb.get('split')!r} — that is a SPLIT comparison, not a backend one. Skipped.")
            continue
        if ma.get("perturbation") == mb.get("perturbation"):
            print(f"[{v}] both runs are {ma.get('perturbation')!r} — nothing to compare. Skipped.")
            continue

        joined = A["avail"][v].attributions.merge(
            B["avail"][v].attributions, on=schema.CLAIM_ID, suffixes=("_i", "_t"))
        feats = [c for c in A["MABS"][v].index
                 if f"{c}_i" in joined.columns and f"{c}_t" in joined.columns]
        if joined.empty or not feats:
            print(f"[{v}] the two runs share no claim, or no feature — skipped.")
            continue

        # SHARES, not unadjusted mean|φ|: the backends need not put the same TOTAL mass on a row
        # (interventional is referenced to a background, path-dependent to the tree's own cover),
        # so an unnormalised difference would be dominated by that scale gap rather than by where
        # the credit sits — which is the only thing being asked here.
        si = conc.shares(A["MABS"][v][feats])
        st = conc.shares(B["MABS"][v][feats])
        corr = [float(np.corrcoef(joined[f"{c}_i"], joined[f"{c}_t"])[0, 1])
                if joined[f"{c}_i"].std() > 0 and joined[f"{c}_t"].std() > 0 else np.nan
                for c in feats]

        t = pd.DataFrame({"share_interventional": si, "share_path_dep": st,
                          "delta_share": si - st, "row_corr": corr}, index=feats)
        t["rank_int"] = t["share_interventional"].rank(ascending=False).astype(int)
        t["rank_tpd"] = t["share_path_dep"].rank(ascending=False).astype(int)
        t["rank_move"] = t["rank_tpd"] - t["rank_int"]        # + = climbs under interventional
        per_feature[v] = t.sort_values("delta_share", ascending=False)

        pi = conc.profile(A["MABS"][v][feats], k=TOPK)
        pt = conc.profile(B["MABS"][v][feats], k=TOPK)
        summary[v] = {
            "n_claims": len(joined),
            "n_features": len(feats),
            "TV_distance": float(0.5 * np.abs(si - st).sum()),
            "spearman_rank": float(t["share_interventional"].corr(t["share_path_dep"],
                                                                  method="spearman")),
            "median_row_corr": float(np.nanmedian(corr)),
            "evenness_D2_D0_int": pi["evenness_D2_D0"],
            "evenness_D2_D0_tpd": pt["evenness_D2_D0"],
            "d_evenness": pi["evenness_D2_D0"] - pt["evenness_D2_D0"],
        }

    if not summary:
        print("no version produced a backend comparison.")
        return None, None

    S = pd.DataFrame(summary).T
    print(f"\n=== {run_int}  vs  {run_tpd} — same split, backend as the only variable ===")
    display(S.round(4))
    # Index is a VERSION, not a feature name -- plain export, no alias twin needed.
    figstyle.save_table(S, f"backend_comparison_summary_{run_int}_vs_{run_tpd}")
    if len(S) > 1 and S["d_evenness"].gt(0).any() and S["d_evenness"].lt(0).any():
        print("⚠ d_evenness changes SIGN across versions — the §3-§5 concentration ordering is "
              "NOT stable under\n  the reference choice. It cannot be reported without this "
              "table beside it.")
    for v, t in per_feature.items():
        print(f"\n[{v}] where credit moves when the reference stops carrying feature dependence "
              f"(top/bottom {top})")
        display(pd.concat([t.head(top), t.tail(top)]).round(4))

        # FULL per-feature table (every feature, not just the top/bottom `top` shown above), real
        # name then alias -- `t` is indexed by v's real encoded feature names, the one exception
        # to this notebook's otherwise version/basis-level tables. See the Notes cell at the bottom.
        figstyle.save_table(t, f"{v}_backend_shift_by_feature_{run_int}_vs_{run_tpd}")
        alias_index = feature_alias.to_alias(v, list(t.index))   # raises if any name is unmapped
        figstyle.save_table(t.set_axis(alias_index, axis=0),
                           f"alias_{v}_backend_shift_by_feature_{run_int}_vs_{run_tpd}")
    return S, per_feature


def _backend_shift_fig(summary: pd.DataFrame, by_feature: dict, aliased: bool) -> None:
    """One scatter panel per version, built fresh — not shared with its real/alias twin, so
    neither figure's layout is sized around the other's label widths (see Notes at the bottom)."""
    vset7 = list(summary.index)
    fig, axes = plt.subplots(1, len(vset7), figsize=(3.6 * len(vset7), 3.5), squeeze=False)
    for ax, v in zip(axes[0], vset7):
        t = by_feature[v]
        lim = float(max(t["share_interventional"].max(), t["share_path_dep"].max())) * 1.12
        ax.plot([0, lim], [0, lim], color=figstyle.NEUTRAL, lw=0.8, zorder=0)
        ax.scatter(t["share_path_dep"], t["share_interventional"], s=18,
                   color=figstyle.PRIMARY, zorder=2)
        for c in t["delta_share"].abs().nlargest(3).index:     # only the movers get a label
            label = feature_alias.to_alias(v, [c])[0] if aliased else c
            ax.annotate(label, (t.loc[c, "share_path_dep"], t.loc[c, "share_interventional"]),
                        fontsize=6, xytext=(3, 3), textcoords="offset points")
        ax.set_xlim(0, lim)
        ax.set_ylim(0, lim)
        ax.set_title(f"{v}   TV={summary.loc[v, 'TV_distance']:.3f}")
        ax.set_xlabel("share · tree_path_dependent")
    axes[0][0].set_ylabel("share · interventional")
    fig.suptitle("Attribution share, interventional vs tree-path-dependent — "
                 "distance from the diagonal = reference sensitivity")
    fig.tight_layout()
    prefix = "alias_" if aliased else ""
    figstyle.save_fig(fig, f"{prefix}07a_real_shap_backend_shift")
    plt.show()


BACKEND_SUMMARY, BACKEND_BY_FEATURE = compare_backends("interventional_oot", "path_dependent")
# BACKEND_SUMMARY, BACKEND_BY_FEATURE = compare_backends("interventional_train",
#                                                        "path_dependent_train")  # see RUN_SPEC

if BACKEND_SUMMARY is not None:
    # TWO INDEPENDENT figures, real name and alias. Real-name PNG is company-laptop/internal
    # only; alias_ is the only version safe to leave the machine.
    _backend_shift_fig(BACKEND_SUMMARY, BACKEND_BY_FEATURE, aliased=False)
    _backend_shift_fig(BACKEND_SUMMARY, BACKEND_BY_FEATURE, aliased=True)

## Notes, and what would invalidate this

- **Two independent PNGs per feature-labelled figure, real name and alias.** §1's `analyze()` and
  §7's `compare_backends()` are the only two figures that label individual features (the y-ticks
  in `01a_real_shap_global_importance_*`, the mover annotations in `07a_real_shap_backend_shift`).
  Each is built by a small helper (`_global_importance_fig()`, `_backend_shift_fig()`) that is
  called TWICE — once with real feature names, once with labels swapped through
  `feature_alias.to_alias(version, names)` — and each call does its own
  `plt.subplots()`/`tight_layout()`/`savefig()`. Two independent figures, not one figure relabelled
  and saved twice, so neither figure's layout is sized around the other's (usually different)
  label widths. The real-name PNG is company laptop / internal use only; the `alias_`-prefixed one
  (`alias_01a_...`, `alias_07a_...`) is the only version safe to leave the machine (thesis, review,
  anywhere the real Allianz column names must not go). §4's Hill curves plot aggregate diversity
  only, so no feature name ever reaches them — one file, no alias twin needed. The mapping itself —
  `features/registry/feature_alias_map.json` — is built once on the company laptop by
  `features/build_feature_alias.py` and never reaches git. `figures/` itself is NOT gitignored, so
  the real-name PNGs are stageable — do not `git add` them.
- **Every table this notebook displays is also saved as CSV** (`figstyle.save_table()`), in FULL
  rather than the `head()`/`round()`-only slice a cell prints. Almost none of them need an alias
  twin, because almost none of them are indexed by a feature name — this notebook works one level
  up, at the concentration-measure level (basis / version / run), and §2's hand-confirmed mapping
  already keeps raw column names out of the comparison basis itself:
    - §2's `feature_overlap_pairs_per_version` and `comparison_bases_summary` — index is a version
      or a basis label.
    - §3's `profile_<run>_<basis>` (one file per run × basis PROFILES produced) and §3b's
      `hyperparameter_comparison` — index is a version or an xgboost hyperparameter name.
    - §3c's `zero_mass_columns_summary` / `zero_mass_columns_in_basis` — index is
      (version, run) / (basis, version, run).
    - §5's `cross_run_concentration_comparison` and §6's `decision_side_profile_<run_label>` —
      index is (basis, version, run) / (basis, version, side).
    - §7's `backend_comparison_summary_<run_int>_vs_<run_tpd>` — index is a version.
  **The one exception is §3c's `zero_mass_columns_list` and §7's
  `<v>_backend_shift_by_feature_<run_int>_vs_<run_tpd>`** — these two ARE indexed (or, for the
  zero-mass list, valued) by real encoded feature names, so each gets the real+alias pair like the
  figures do: §3c's zero-mass column names (only ever printed truncated to 30 before) go out
  long-format (`version, run, feature`) and are aliased per version-group
  (`feature_alias.to_alias(v, ...)`, since each version has its own map); §7's per-feature table
  (`t` in `compare_backends()`, previously only shown as its top/bottom-`top` slice) is aliased by
  replacing its index outright (`t.set_axis(feature_alias.to_alias(v, list(t.index)), axis=0)`,
  which raises rather than silently passing a name through if the map is stale).
- **Three configurations, one figure set.** `interventional_train` / `interventional_oot` /
  `path_dependent` (§0) are not alternatives to pick between — the point of this notebook version is
  that all three run together and §5 puts them in one table. A large move between
  `interventional_train` and `interventional_oot` says the concentration finding is split-sensitive;
  a large move between `interventional_oot` and `path_dependent` says it is backend-sensitive. Either
  weakens the finding; neither is fatal on its own, but both must be reported alongside it.
- **Backend.** If a run reports `perturbation = tree_path_dependent` (always true for
  `path_dependent`), the reference is each tree's own cover statistics, so part of any
  cross-version difference under that run is a difference in training distributions, not only in
  the fitted functions. `interventional_*` uses a fixed shared background instead — prefer it, and
  keep `path_dependent` as the robustness check it is here, not the headline number.
- **Rows.** No claim is common to all three versions — v2 runs 2018-01→2020-09 and v3 runs
  2023-06→2026-05, ~2¾ years apart. So each version is attributed **per version by default**, and
  every difference here is confounded with case-mix. That caveat is standing, not conditional.
  A hand-built claim_id intersection, passed to attribute.py via --explain-ids/--background-ids,
  is only possible for a pair that overlaps in time (v1/v2) — refuse to proceed on an empty
  intersection rather than silently sampling.
- **Windows.** v2 and v3 were trained years apart (README), so concentration differences are
  *descriptive* of the fitted functions, not evidence of the loop on their own — the
  identification argument is the DiD in `04_02`, and its parallel-trends assumption is the thing
  to defend.
- **`config.path()` is still keyed by `(version, split)` only.** `--out-suffix` is a notebook-local
  workaround for comparing backends on one split, not a general second axis — it exists on
  the caller's own `--out` path but nowhere in `config.py`/`loaders.py`, so nothing downstream of this
  notebook resolves a suffixed file automatically. If a suffixed run becomes a permanent fixture
  rather than a one-off robustness check, promote it to a real `config.SPLIT_KINDS`-style axis
  instead of growing more of these local readers.
- **The mapping is the only bridge.** If §2 found no mapping file, `BASES` holds only the 1-way
  `(own features)` bases and no restricted number exists to report. And if a future edit ever ranks `make` rather than `make_FORD`, the
  number has changed meaning — collapse never happens here; correspondence comes only from
  `features/feature_overlap.json` (hand-confirmed; typo-checked against the registries by
  `features/check_overlap.py`).